# Phase 10 — locked transfer to an independent reporter assay

**Run on Kaggle:** GPU T4/P100 or better, Internet ON. Attach three saved Kaggle Inputs: (1) `9_enhancer_phase9a_full_activity_dataset.zip` (or its unpacked contents); (2) Phase 9B `enhancer_phase9b_complete_with_weights.zip` or the saved Phase 9B Output with its results and `checkpoints` folder; (3) the equivalent Phase 9C complete bundle or saved Output. The small 9B/9C results ZIPs alone have no model weights. The public 31.7 MB external table downloads automatically. If Springer/PMC cannot be reached from Kaggle, upload `41586_2024_8070_MOESM14_ESM.txt` as a fourth Input; its SHA-256 is pinned below. **Do not attach unrelated earlier model weights.**

**Source:** Gosai et al., *Nature* 2024, DOI [10.1038/s41586-024-08070-z](https://doi.org/10.1038/s41586-024-08070-z), Supplementary Table 12: [CODA library experimental MPRA results](https://pmc.ncbi.nlm.nih.gov/articles/PMC11525185/#MOESM14). This is a separate episomal reporter library assayed in HepG2 and K562. The Phase 9 training targets came from an ENCODE lentiMPRA release. We evaluate sequence ranking across assays; their activity scales are not equated. Some CODA elements were deliberately designed with a different model, so **`DHS_natural` is the primary stratum**; `Malinois_natural` and three synthetic origins are reported separately.

**Frozen protocol:** retain exactly 200 A/C/G/T bases, complete measured activity and standard error ≤1 in both cells. For natural sequences, parse hg38 intervals embedded in Table 12 IDs and remove every interval overlapping any Phase 9A interval. Also remove all exact/reverse-complement matches and any shared exact 31-mer with *all* Phase 9A rows (train, validation and test, both cells). Collapse exact/reverse-complement duplicates. Sample exactly 4,000 elements per synthetic design origin by sequence SHA-256, independent of activity. The natural groups use every eligible element. A shared 31-mer is a conservative contamination screen; its absence does not prove absence of every distant homology. Synthetic sequences have no genomic coordinates.

**Models:** independently replay Phase 9B chromosome and Phase 9C locus frozen GC/CpG and spaced-6/4-mer ridge baselines using the original training rows and the validation-selected ridge alphas; load all 12 saved CNN and DNABERT-2 checkpoints **per protocol**. No parameters are fitted to CODA measurements. Report Spearman as the primary cross-assay statistic; Pearson as secondary. Include the original K562 Phase 9C seed `20260930` even though it collapsed on internal validation. All three-seed means remain prespecified. Confidence intervals compare matched predictions with 300 paired bootstrap draws on unique external sequences. No R²/RMSE/MAE are reported across assay scales.

Run **all** cells in order. Jobs save individually under `/kaggle/working/enhancer_phase10_work` and can resume from a saved Phase 10 notebook Output attached as an additional Input. Keep the final `enhancer_phase10_external_transfer_results.zip`; the notebook writes prediction rows, inclusion audit, metrics and provenance. Do not choose models, subset thresholds or response transformations after viewing the external scores.


In [ ]:
# 0 — Locked inputs, provenance and resumable directory.
import os, sys, io, json, zipfile, hashlib, random, gc, shutil
import subprocess, urllib.request
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfTransformer

ROOT = Path('/kaggle/input')
assert ROOT.is_dir(), 'Attach the three saved Phase 9 Inputs described above.'

WORK = Path('/kaggle/working/enhancer_phase10_work')
WORK.mkdir(parents=True, exist_ok=True)
SCORED = WORK / 'scores'
SCORED.mkdir(exist_ok=True)
TMP = WORK / 'current_checkpoint.pt'

INPUT_HASH = {
    'phase9a_audit_and_provenance.json':
        '39118ed7b2dc50631f87dcb0c470249c649609d3dd7ab8c885b10064c66deff2',
    'phase9_full_activity_all_cells.csv':
        '4a359a715f7a3f1fbcf0d044cc08446137df584c9ab4a43a2de18637bb341a76',
    'phase9_frozen_splits.csv':
        '1ecc2d271817edd3000890be718a3f3a9bcc6b63a71550dfa718dbed84d0ff61'
}
EXTERNAL_HASH = (
    '1dcc68b0340933acf593c6bbbc8a8ce2d7aaae60334eebf353958a0f0459fdae'
)
SOURCE_URLS = [
    'https://pmc.ncbi.nlm.nih.gov/articles/instance/11525185/bin/'
    '41586_2024_8070_MOESM14_ESM.txt',
    'https://media.springernature.com/original/springer-static/esm/'
    'art%3A10.1038%2Fs41586-024-08070-z/MediaObjects/'
    '41586_2024_8070_MOESM14_ESM.txt'
]
LOCKS = {
    '9b': '9a5c05aca44b1007301965ad5d7c7877629865071ad6fae88b2692d3a9cd199d',
    '9c': 'dd2b4c43281df1782900f5b11265b80d1539c635ee261fe82d07f70a8705f610'
}
REVISION = 'ec1f874253852eb3907081f57294991b4280ceb6'

CONFIG = {
    'name': 'phase10_gosai_coda_external_v1',
    'source_sha256': EXTERNAL_HASH,
    'phase9a_sha256': INPUT_HASH,
    'phase9_locks': LOCKS,
    'external_length': 200,
    'standard_error_max': 1.0,
    'shared_exact_k': 31,
    'natural_coordinate_policy':
        'hg38 any interval intersection with any Phase 9A record across cells excluded',
    'primary_origin': 'DHS_natural',
    'secondary_origins': [
        'Malinois_natural', 'AdaLead', 'FastSeqProp',
        'Simulated_Annealing'
    ],
    'synthetic_per_origin': 4000,
    'selection': 'first 4000 by SHA256 of canonical DNA after QC',
    'metrics': ['Spearman', 'Pearson'],
    'bootstrap_seed': 20261014,
    'bootstrap_repeats': 300,
    'cnn_batch': 128,
    'dnabert2_max_tokens': 256,
    'model_policy':
        'both Phase 9 protocols, all 3 declared seeds and declared means; no external selection',
    'source_note': 'Table 12 of DOI 10.1038/s41586-024-08070-z'
}

def sha(data):
    return hashlib.sha256(data).hexdigest()

def fsha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

LOCK = sha(json.dumps(
    CONFIG, sort_keys=True, separators=(',', ':')
).encode())
print('Phase 10 protocol:', LOCK)

ZIPS = sorted(p for p in ROOT.rglob('*.zip') if p.is_file())

def member_sources(name):
    for path in ROOT.rglob(name):
        if path.is_file():
            yield ('file', path, None)

    for path in ZIPS:
        try:
            with zipfile.ZipFile(path) as archive:
                for member in archive.namelist():
                    if member == name or member.endswith('/' + name):
                        yield ('zip', path, member)
        except zipfile.BadZipFile:
            continue

def read_source(source):
    kind, path, member = source
    if kind == 'file':
        return path.read_bytes()
    with zipfile.ZipFile(path) as archive:
        return archive.read(member)

def read_locked(name, expected):
    hits = []
    for source in member_sources(name):
        raw = read_source(source)
        if sha(raw) == expected:
            hits.append((raw, source))

    if not hits:
        raise FileNotFoundError(
            'Missing or wrong-version ' + name +
            '; attach the exact Phase 9A saved Output.'
        )

    assert len({sha(raw) for raw, _ in hits}) == 1
    print('Verified', name, 'from', hits[0][1][1])
    return hits[0][0]

ra = read_locked(
    'phase9a_audit_and_provenance.json',
    INPUT_HASH['phase9a_audit_and_provenance.json']
)
rd = read_locked(
    'phase9_full_activity_all_cells.csv',
    INPUT_HASH['phase9_full_activity_all_cells.csv']
)
rs = read_locked(
    'phase9_frozen_splits.csv',
    INPUT_HASH['phase9_frozen_splits.csv']
)

audit9a = json.loads(ra)
assert audit9a['protocol_hash'] == (
    '5e438773a3f9c74cf006d183bcfc30daabd3d85bdbfefd47acfa48dc023e450a'
)

phase9 = pd.read_csv(
    io.BytesIO(rd),
    dtype={'record_id': str, 'sequence': str, 'cell_line': str}
)
frozen = pd.read_csv(io.BytesIO(rs), dtype=str, keep_default_na=False)

assert len(phase9) == len(frozen) == 76833
assert phase9.record_id.tolist() == frozen.record_id.tolist()
assert phase9.sequence.str.fullmatch('[ACGT]{200}').all()

for column in ('split_9a', 'split_9b'):
    assert (
        phase9[column].astype(str).tolist() ==
        frozen[column].tolist()
    )

assert phase9.groupby('split_group_id').split_9a.nunique().max() == 1
assert phase9.groupby('locus_group_id').split_9b.nunique().max() == 1

# Kaggle may expose a direct results ZIP, a ZIP nested in a weights bundle,
# or the result members already unpacked into individual files.
def result_bytes(tag):
    provenance_name = f'phase{tag}_provenance.json'
    seen = []
    hits = []

    def check(raw, location):
        try:
            with zipfile.ZipFile(io.BytesIO(raw)) as archive:
                names = archive.namelist()
                matches = [
                    member for member in names
                    if Path(member).name == provenance_name
                ]
                if len(matches) != 1:
                    return

                report = json.loads(archive.read(matches[0]))
                if report.get('protocol_sha256') != LOCKS[tag]:
                    seen.append((
                        location,
                        'wrong protocol ' +
                        str(report.get('protocol_sha256'))
                    ))
                    return

                for name, digest in report['result_sha256'].items():
                    members = [
                        member for member in names
                        if Path(member).name == name
                    ]
                    if (
                        len(members) != 1 or
                        sha(archive.read(members[0])) != digest
                    ):
                        raise RuntimeError(
                            f'Phase {tag.upper()} result member failed '
                            f'SHA256: {location}!{name}'
                        )

                hits.append((raw, location))
        except (zipfile.BadZipFile, KeyError, ValueError):
            return

    # Inspect mounted ZIPs by their contents, regardless of filename.
    for path in ZIPS:
        try:
            with zipfile.ZipFile(path) as archive:
                names = archive.namelist()

                if any(
                    Path(member).name == provenance_name
                    for member in names
                ):
                    check(path.read_bytes(), str(path))

                for member in names:
                    if member.lower().endswith('.zip'):
                        check(
                            archive.read(member),
                            str(path) + '!' + member
                        )
        except (zipfile.BadZipFile, OSError):
            continue

    if hits:
        fingerprints = {sha(raw) for raw, _ in hits}
        assert len(fingerprints) == 1, (
            f'Conflicting Phase {tag.upper()} result archives: '
            f'{[location for _, location in hits]}'
        )
        print(
            'Verified Phase', tag.upper(),
            'result ZIP from', hits[0][1]
        )
        return hits[0][0]

    # Rebuild an in-memory results ZIP when Kaggle unpacked its members.
    # Each member must match the hash in the original provenance report.
    for provenance_path in ROOT.rglob(provenance_name):
        if not provenance_path.is_file():
            continue

        report_raw = provenance_path.read_bytes()
        report = json.loads(report_raw)

        if report.get('protocol_sha256') != LOCKS[tag]:
            seen.append((str(provenance_path), 'wrong protocol'))
            continue

        members = {provenance_name: report_raw}
        for name, digest in report['result_sha256'].items():
            found = [
                path for path in provenance_path.parent.rglob(name)
                if path.is_file() and fsha(path) == digest
            ]
            if not found:
                raise FileNotFoundError(
                    f'Found {provenance_path}, but verified result '
                    f'member {name} is missing.'
                )
            members[name] = found[0].read_bytes()

        buffer = io.BytesIO()
        with zipfile.ZipFile(
            buffer, 'w', compression=zipfile.ZIP_DEFLATED
        ) as archive:
            for name, raw in members.items():
                archive.writestr(name, raw)

        check(
            buffer.getvalue(),
            'rebuilt from ' + str(provenance_path.parent)
        )

    if hits:
        print(
            'Verified unpacked Phase', tag.upper(),
            'results from', hits[0][1]
        )
        return hits[0][0]

    nearby = [
        str(path.relative_to(ROOT))
        for path in ROOT.rglob('*')
        if f'phase{tag}' in str(path).lower()
    ][:50]
    raise FileNotFoundError(
        f'No Phase {tag.upper()} results found; '
        f'relevant mounted paths: {nearby}; inspected: {seen}'
    )

REPORT = {}
CHOICES = {}
RESULT_SHA = {}

for tag in LOCKS:
    raw = result_bytes(tag)
    RESULT_SHA[tag] = sha(raw)

    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        assert archive.testzip() is None
        report = json.loads(
            archive.read(f'phase{tag}_provenance.json')
        )

        assert report['protocol_sha256'] == LOCKS[tag]
        assert report['config']['split'] == (
            'split_9b' if tag == '9b' else 'split_9a'
        )
        assert report['phase9a_data_sha256'] == sha(rd)
        assert report['phase9a_split_sha256'] == sha(rs)
        assert report['config']['dnabert2_revision'] == REVISION

        for name, digest in report['result_sha256'].items():
            assert sha(archive.read(name)) == digest

        choices = pd.read_csv(io.BytesIO(
            archive.read(f'phase{tag}_validation_choices.csv')
        ))
        assert len(choices) == 16
        assert not choices.duplicated(['cell_line', 'model']).any()
        assert len(report['checkpoints']) == 12

        REPORT[tag] = report
        CHOICES[tag] = choices

    print(
        'VERIFIED Phase', tag.upper(), 'results',
        RESULT_SHA[tag], 'and 12 declared checkpoint hashes'
    )

for source in ROOT.rglob('*'):
    if (
        source.is_file() and
        source.parent.name == 'scores' and
        source.parent.parent.name == 'enhancer_phase10_work'
    ):
        destination = SCORED / source.name
        if destination.exists():
            assert fsha(destination) == fsha(source)
        else:
            shutil.copy2(source, destination)

print('Prior Phase 10 score files:', len(list(SCORED.iterdir())))

Phase 10 protocol: 0de96f349bf5bc6342b45e449d1833ef397ea467a00f3e6b1d55f7371e9a7fa8
Verified phase9a_audit_and_provenance.json from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9a_audit_and_provenance.json
Verified phase9_full_activity_all_cells.csv from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9_full_activity_all_cells.csv
Verified phase9_frozen_splits.csv from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9_frozen_splits.csv
Verified unpacked Phase 9B results from rebuilt from /kaggle/input/datasets/aryanaditta/9b-enhancer-phase9b-regression-results-zip
VERIFIED Phase 9B results 6a4314542f12a47dae050c1c3f139838d75f19d36193c53b59cc05d426bdf642 and 12 declared checkpoint hashes
Verified unpacked Phase 9C results from rebuilt from /kaggle/input/datasets/aryanaditta/9c-enhancer-phase9c-complete-with-weights-zip/enhancer_phase9c_regression_results
VERIFIED Phase 9C re

In [ ]:
# 1 — Acquire original CODA Table 12 and freeze contamination/QC screen BEFORE inference.
SOURCE=WORK/'41586_2024_8070_MOESM14_ESM.txt'
if SOURCE.exists():assert fsha(SOURCE)==EXTERNAL_HASH
else:
 attached=[p for p in ROOT.rglob(SOURCE.name) if p.is_file() and fsha(p)==EXTERNAL_HASH]
 if attached:shutil.copy2(attached[0],SOURCE)
 else:
  for archive in ZIPS:
   with zipfile.ZipFile(archive) as z:
    members=[m for m in z.namelist() if Path(m).name==SOURCE.name]
    if len(members)==1:
     candidate=z.read(members[0])
     if sha(candidate)==EXTERNAL_HASH:
      SOURCE.write_bytes(candidate);print('Verified attached CODA source ZIP:',archive);break
 if not SOURCE.exists():
  errors=[]
  for url in SOURCE_URLS:
   try:
    tmp=SOURCE.with_suffix('.part')
    with urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'}),timeout=180) as response,open(tmp,'wb') as f:
     shutil.copyfileobj(response,f)
    if fsha(tmp)!=EXTERNAL_HASH:raise RuntimeError('External source SHA256 differs from frozen table')
    os.replace(tmp,SOURCE);print('Downloaded pinned Table 12 from',url);break
   except Exception as e:
    errors.append(repr(e))
    if tmp.exists():tmp.unlink()
  if not SOURCE.exists():raise RuntimeError('Attach '+SOURCE.name+' as Kaggle Input; both publisher mirrors failed: '+str(errors))
assert fsha(SOURCE)==EXTERNAL_HASH
raw=pd.read_csv(SOURCE,sep='\t',low_memory=False,dtype={'ID':str,'sequence':str,'origin':str,'target_cell':str})
EXPECTED_COLUMNS={'ID','sequence','origin','target_cell','K562_l2fc','HepG2_l2fc','K562_lfcSE','HepG2_lfcSE'}
assert EXPECTED_COLUMNS.issubset(raw.columns) and len(raw)==77157 and raw.ID.is_unique
AUDIT={'source_rows':len(raw),'source_sha256':EXTERNAL_HASH}
raw=raw[raw.sequence.str.fullmatch('[ACGT]{200}',na=False)].copy()
AUDIT['canonical_200bp_rows']=len(raw)
ORIGINS=['DHS_natural','Malinois_natural','AdaLead','FastSeqProp','Simulated_Annealing']
raw=raw[raw.origin.isin(ORIGINS)].copy();AUDIT['chosen_origins_rows']=len(raw)
for col in ('K562_l2fc','HepG2_l2fc','K562_lfcSE','HepG2_lfcSE'):
 raw[col]=pd.to_numeric(raw[col],errors='coerce')
good=np.isfinite(raw[['K562_l2fc','HepG2_l2fc','K562_lfcSE','HepG2_lfcSE']]).all(axis=1)
good &= raw.K562_lfcSE.le(CONFIG['standard_error_max']) & raw.HepG2_lfcSE.le(CONFIG['standard_error_max'])
raw=raw[good].copy();AUDIT['both_cells_activity_and_se_pass']=len(raw)

# The paper specifies GRCh38/hg38, and both natural ID families embed intervals.
from bisect import bisect_left
natural_mask=raw.origin.isin(ORIGINS[:2])
coords=raw.loc[natural_mask,'ID'].str.extract(r'(chr(?:[1-9]|1[0-9]|2[0-2]|X|Y)):([0-9]+)-([0-9]+)')
assert coords.notna().all().all(),'A natural source ID lacks its declared hg38 interval.'
raw['external_chrom']='';raw['external_start']=-1;raw['external_end']=-1
raw.loc[natural_mask,'external_chrom']=coords[0].to_numpy()
raw.loc[natural_mask,'external_start']=coords[1].astype(int).to_numpy()
raw.loc[natural_mask,'external_end']=coords[2].astype(int).to_numpy()
bad_interval=natural_mask & (raw.external_end-raw.external_start).ne(200)
AUDIT['natural_metadata_non_200bp_intervals']=int(bad_interval.sum())
raw=raw[~bad_interval].copy()
natural_mask=raw.origin.isin(ORIGINS[:2])
chrom_intervals={}
for chrom,group in phase9.groupby('chrom'):
 ordered=group.sort_values('start')
 chrom_intervals[chrom]=(ordered.start.to_numpy(dtype=np.int64),
  np.maximum.accumulate(ordered.end.to_numpy(dtype=np.int64)))
def intersects_phase9(chrom,start,end):
 if chrom not in chrom_intervals:return False
 starts,prefix_max_end=chrom_intervals[chrom]
 i=np.searchsorted(starts,end,side='left')-1
 return i>=0 and prefix_max_end[i]>start
natural=raw[natural_mask]
overlap=[intersects_phase9(c,s,e) for c,s,e in zip(natural.external_chrom,natural.external_start,natural.external_end)]
raw['phase9_interval_overlap']=False
raw.loc[natural_mask,'phase9_interval_overlap']=overlap
AUDIT['natural_interval_overlap_by_origin']=raw.groupby('origin').phase9_interval_overlap.sum().astype(int).to_dict()
raw=raw[~raw.phase9_interval_overlap].copy()
AUDIT['rows_after_natural_interval_screen']=len(raw)

# Two-bit rolling canonical 31-mer, shared with its reverse complement.
K=CONFIG['shared_exact_k'];MASK=(1<<(2*K))-1;SHIFT=2*(K-1)
BASE={65:0,67:1,71:2,84:3}
def kmers(s):
 f=r=0
 for i,b in enumerate(s.encode()):
  x=BASE[b];f=((f<<2)|x)&MASK;r=(r>>2)|((3-x)<<SHIFT)
  if i>=K-1:yield min(f,r)
train_words=np.empty(len(phase9)*(201-K),dtype=np.uint64)
for i,s in enumerate(phase9.sequence):train_words[i*(201-K):(i+1)*(201-K)]=list(kmers(s))
train_words=np.unique(train_words)
AUDIT['phase9_unique_canonical_31mers']=int(len(train_words))
screen=[]
for offset in range(0,len(raw),500):
 chunk=raw.iloc[offset:offset+500]
 query=np.asarray([word for seq in chunk.sequence for word in kmers(seq)],dtype=np.uint64)
 ix=np.searchsorted(train_words,query)
 hit=(ix<len(train_words)) & (train_words[np.minimum(ix,len(train_words)-1)]==query)
 screen.extend(hit.reshape(len(chunk),201-K).any(axis=1).tolist())
raw['shared_phase9_31mer']=screen
AUDIT['shared_31mer_by_origin']=raw.groupby('origin').shared_phase9_31mer.sum().astype(int).to_dict()
raw=raw[~raw.shared_phase9_31mer].copy()
AUDIT['rows_after_31mer_screen']=len(raw)
del train_words;gc.collect()
TRANS=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(TRANS)[::-1]
raw['canonical']=raw.sequence.map(lambda s:min(s,rc(s)))
AUDIT['cross_origin_exact_rc_duplicates']=int(raw.duplicated('canonical').sum())
priority={v:i for i,v in enumerate(ORIGINS)}
raw['origin_priority']=raw.origin.map(priority)
raw=raw.sort_values(['origin_priority','ID']).drop_duplicates('canonical',keep='first').copy()
natural=raw[raw.origin.isin(ORIGINS[:2])]
synthetic=pd.concat([g.assign(sample_order=g.canonical.map(lambda s:sha(s.encode()))).sort_values(['sample_order','ID']).head(CONFIG['synthetic_per_origin'])
 for origin,g in raw[raw.origin.isin(ORIGINS[2:])].groupby('origin')],ignore_index=True)
external=pd.concat([natural,synthetic],ignore_index=True).sort_values(['origin_priority','ID']).reset_index(drop=True)
external['record_id']=external.canonical.map(lambda s:sha(s.encode())[:20])
assert external.record_id.is_unique and external.canonical.is_unique and external.ID.is_unique
assert set(external.origin)==set(ORIGINS)
assert all((external.origin==o).sum()==4000 for o in ORIGINS[2:])
assert (external.origin=='DHS_natural').sum()>=5000
assert np.isfinite(external[['K562_l2fc','HepG2_l2fc']]).all().all()
AUDIT['frozen_counts_by_origin']=external.origin.value_counts().to_dict()
cols=['record_id','ID','sequence','origin','target_cell','round','track_ID','K562_l2fc','HepG2_l2fc','K562_lfcSE','HepG2_lfcSE','canonical','external_chrom','external_start','external_end']
frozen_external=WORK/'phase10_frozen_external.csv'
external[cols].to_csv(frozen_external,index=False,float_format='%.12g')
FROZEN_SHA=fsha(frozen_external)
AUDIT['frozen_csv_sha256']=FROZEN_SHA
(WORK/'phase10_inclusion_audit.json').write_text(json.dumps(AUDIT,indent=2,sort_keys=True)+'\n')
print('FROZEN',len(external),'rows',external.origin.value_counts().to_dict(),'SHA256',FROZEN_SHA)


FROZEN 29330 rows {'Malinois_natural': 8931, 'DHS_natural': 8399, 'AdaLead': 4000, 'FastSeqProp': 4000, 'Simulated_Annealing': 4000} SHA256 00d3521b5f087df56e97b792e7b961ffcb9a80e346fdb4c8360dc0182dc273f8


In [ ]:
# 2 — Per-model locked score storage and authenticated checkpoint access.
PROTOCOLS=('9b','9c');CELLS=('HepG2','K562');SEEDS=(20260930,20260931,20260932)
BASE_ARMS=('gc_cpg_ridge','spaced6_4_ridge')
NEURAL_ARMS=tuple(f'{f}_seed_{s}' for f in ('cnn','dnabert2') for s in SEEDS)
ARMS=BASE_ARMS+NEURAL_ARMS
def choice(tag,cell,arm):
 x=CHOICES[tag];y=x[(x.cell_line==cell)&(x.model==arm)]
 assert len(y)==1,(tag,cell,arm)
 return y.iloc[0]
def score_path(tag,cell,arm):return SCORED/f'phase{tag}_{cell}_{arm}.csv'
def score_meta(tag,cell,arm):return SCORED/f'phase{tag}_{cell}_{arm}.json'
def checkpoint_expected(tag,cell,arm):
 name=f'{cell}_{arm}_best.pt'
 assert name in REPORT[tag]['checkpoints'],(tag,name)
 return name,REPORT[tag]['checkpoints'][name]['sha256']
def done(tag,cell,arm):
 p=score_path(tag,cell,arm);m=score_meta(tag,cell,arm)
 if not (p.exists() and m.exists()):return None
 info=json.loads(m.read_text())
 if info.get('phase10_lock')!=LOCK or info.get('external_sha256')!=FROZEN_SHA:return None
 assert info['prediction_sha256']==fsha(p)
 x=pd.read_csv(p,dtype={'record_id':str,'score':float})
 assert x.record_id.tolist()==external.record_id.tolist() and np.isfinite(x.score).all()
 if arm in NEURAL_ARMS:
  assert info['checkpoint_sha256']==checkpoint_expected(tag,cell,arm)[1]
 return x.score.to_numpy()
def save_score(tag,cell,arm,values,checkpoint_sha256=None):
 assert len(values)==len(external) and np.isfinite(values).all()
 p=score_path(tag,cell,arm);tmp=p.with_suffix('.part')
 pd.DataFrame({'record_id':external.record_id,'score':values}).to_csv(tmp,index=False,float_format='%.12g')
 os.replace(tmp,p)
 info={'phase10_lock':LOCK,'external_sha256':FROZEN_SHA,'phase9_protocol':LOCKS[tag],
  'checkpoint_sha256':checkpoint_sha256,'prediction_sha256':fsha(p)}
 t=score_meta(tag,cell,arm).with_suffix('.part');t.write_text(json.dumps(info,sort_keys=True)+'\n');os.replace(t,score_meta(tag,cell,arm))
 assert done(tag,cell,arm) is not None

# Kaggle may expose checkpoint files directly, or leave them in an outer ZIP.
CHECK_CATALOG={}
for p in ROOT.rglob('*.pt'):
 if p.is_file():CHECK_CATALOG.setdefault(p.name,[]).append(('file',p,None))
for p in ZIPS:
 try:
  with zipfile.ZipFile(p) as z:
   for name in z.namelist():
    if name.endswith('.pt'):CHECK_CATALOG.setdefault(Path(name).name,[]).append(('zip',p,name))
 except zipfile.BadZipFile:pass
for tag in PROTOCOLS:
 for name in REPORT[tag]['checkpoints']:
  assert CHECK_CATALOG.get(name),'Missing '+name+'; attach the Phase '+tag.upper()+' complete weights bundle or saved Output.'
def checkpoint_file(tag,cell,arm):
 name,expected=checkpoint_expected(tag,cell,arm)
 options=sorted(CHECK_CATALOG[name],key=lambda src:(f'phase{tag}' not in str(src[1]).lower(),src[0]!='file'))
 for kind,p,member in options:
  if kind=='file':
   if fsha(p)==expected:return p,False,expected
  else:
   if TMP.exists():TMP.unlink()
   with zipfile.ZipFile(p) as z,z.open(member) as stream,open(TMP,'wb') as out:shutil.copyfileobj(stream,out)
   if fsha(TMP)==expected:return TMP,True,expected
   TMP.unlink()
 raise RuntimeError('No checkpoint matches frozen SHA256 for '+tag+' '+name)


In [ ]:
# 3 — Phase 9 classical baselines, refitted only on the same Phase 9 training rows.
from itertools import combinations
MASKS=np.asarray(list(combinations(range(6),4)),dtype=np.int16)
WEIGHT=np.array([64,16,4,1],dtype=np.int32)
BASE_MAP=np.zeros(256,dtype=np.uint8)
for base,num in zip(b'ACGT',range(4)):BASE_MAP[base]=num
def feature_gc(seqs):
 return np.asarray([[(s.count('G')+s.count('C'))/200.,s.count('CG')/199.] for s in seqs],dtype=np.float32)

def feature_spaced(seqs):
 # Fixed, interpretable 15-mask x 256-word vocabulary; no labels used here.
 rows=[]
 for start in range(0,len(seqs),256):
  batch=np.zeros((min(256,len(seqs)-start),3840),dtype=np.float32)
  for j,seq in enumerate(seqs[start:start+256]):
   values=BASE_MAP[np.frombuffer(seq.encode(),dtype=np.uint8)]
   for strand in (values,3-values[::-1]):
    windows=np.lib.stride_tricks.sliding_window_view(strand,6)
    words=(windows[:,MASKS].astype(np.int32)*WEIGHT).sum(-1)
    words+=256*np.arange(15,dtype=np.int32)[None,:]
    batch[j]+=np.bincount(words.ravel(),minlength=3840).astype(np.float32)
  rows.append(sparse.csr_matrix(batch))
 return sparse.vstack(rows,format='csr')

def train_frame(tag,cell):
 split='split_9b' if tag=='9b' else 'split_9a'
 return phase9[(phase9.cell_line==cell)&(phase9[split]=='train')].reset_index(drop=True)
def external_spaced():
 path=WORK/'external_spaced6_4.npz';m=WORK/'external_spaced6_4_meta.json'
 if path.exists() and m.exists():
  info=json.loads(m.read_text())
  assert info=={'lock':LOCK,'source':FROZEN_SHA,'sha256':fsha(path)}
  return sparse.load_npz(path)
 x=feature_spaced(external.sequence)
 tmp=path.with_suffix('.part.npz');sparse.save_npz(tmp,x);os.replace(tmp,path)
 m.write_text(json.dumps({'lock':LOCK,'source':FROZEN_SHA,'sha256':fsha(path)},sort_keys=True)+'\n')
 return x
for tag in PROTOCOLS:
 for cell in CELLS:
  tr=train_frame(tag,cell);y=tr.mean_log2_activity.to_numpy(dtype=float)
  for arm in BASE_ARMS:
   if done(tag,cell,arm) is not None:
    print('REUSED baseline',tag,cell,arm,flush=True);continue
   alpha=float(choice(tag,cell,arm).selected_alpha)
   assert alpha in REPORT[tag]['config']['classical_alphas']['gc_cpg' if arm=='gc_cpg_ridge' else 'spaced6_4']
   if arm=='gc_cpg_ridge':
    a=feature_gc(tr.sequence);b=feature_gc(external.sequence)
    transformer=StandardScaler().fit(a)
   else:
    a=feature_spaced(tr.sequence);b=external_spaced()
    transformer=TfidfTransformer(norm='l2').fit(a)
   a=transformer.transform(a);b=transformer.transform(b)
   model=Ridge(alpha=alpha,solver='lsqr' if sparse.issparse(a) else 'auto').fit(a,y)
   save_score(tag,cell,arm,model.predict(b))
   print('SAVED baseline',tag,cell,arm,'alpha',alpha,flush=True)
   del a,b,transformer,model;gc.collect()


REUSED baseline 9b HepG2 gc_cpg_ridge
REUSED baseline 9b HepG2 spaced6_4_ridge
REUSED baseline 9b K562 gc_cpg_ridge
REUSED baseline 9b K562 spaced6_4_ridge
REUSED baseline 9c HepG2 gc_cpg_ridge
REUSED baseline 9c HepG2 spaced6_4_ridge
REUSED baseline 9c K562 gc_cpg_ridge
REUSED baseline 9c K562 spaced6_4_ridge


In [ ]:
# 4 — Frozen Phase 9 one-hot CNN weights, no training.
import torch
from torch import nn
assert torch.cuda.is_available(),'Select a Kaggle GPU accelerator.'
DEVICE=torch.device('cuda:0')
class CNN(nn.Module):
 def __init__(self):
  super().__init__()
  self.conv=nn.Sequential(nn.Conv1d(4,64,15,padding=7),nn.GELU(),
   nn.Conv1d(64,96,7,padding=3),nn.GELU(),nn.Conv1d(96,96,7,padding=3),nn.GELU())
  self.dropout=nn.Dropout(.2);self.head=nn.Sequential(nn.Linear(192,64),nn.GELU(),nn.Linear(64,1))
 def forward(self,x):
  h=self.conv(x);pooled=torch.cat([h.mean(-1),h.amax(-1)],dim=1)
  return self.head(self.dropout(pooled)).squeeze(-1)

def onehot(seqs):
 codes=np.stack([BASE_MAP[np.frombuffer(s.encode(),dtype=np.uint8)] for s in seqs])
 eye=np.eye(4,dtype=np.float32)
 return torch.from_numpy(eye[codes].transpose(0,2,1).copy()).to(DEVICE)

def cnn_predict(model,seqs,mean,scale):
 model.eval();pred=[]
 with torch.inference_mode():
  for i in range(0,len(seqs),CONFIG['cnn_batch']):
   batch=seqs[i:i+CONFIG['cnn_batch']]
   p=model(onehot(batch)).float();q=model(onehot([rc(s) for s in batch])).float()
   pred.extend((((p+q)/2)*scale+mean).cpu().numpy())
 return np.asarray(pred,dtype=float)

for tag in PROTOCOLS:
 for cell in CELLS:
  for seed in SEEDS:
   arm=f'cnn_seed_{seed}'
   if done(tag,cell,arm) is not None:
    print('REUSED CNN',tag,cell,seed,flush=True);continue
   path,temporary,digest=checkpoint_file(tag,cell,arm)
   ckpt=torch.load(path,map_location='cpu',weights_only=True)
   assert ckpt['lock']==LOCKS[tag] and ckpt['seed']==seed and ckpt['cell_line']==cell
   assert ckpt['architecture']=='cnn_v1'
   if temporary:path.unlink()
   net=CNN().to(DEVICE);net.load_state_dict(ckpt['state_dict'],strict=True)
   values=cnn_predict(net,external.sequence.tolist(),ckpt['target_mean'],ckpt['target_scale'])
   save_score(tag,cell,arm,values,digest)
   print('SAVED CNN',tag,cell,seed,flush=True)
   del values,ckpt,net;gc.collect();torch.cuda.empty_cache()


REUSED CNN 9b HepG2 20260930
REUSED CNN 9b HepG2 20260931
REUSED CNN 9b HepG2 20260932
REUSED CNN 9b K562 20260930
REUSED CNN 9b K562 20260931
REUSED CNN 9b K562 20260932
REUSED CNN 9c HepG2 20260930
REUSED CNN 9c HepG2 20260931
REUSED CNN 9c HepG2 20260932
REUSED CNN 9c K562 20260930
REUSED CNN 9c K562 20260931
REUSED CNN 9c K562 20260932


In [ ]:
# 5 — Frozen DNABERT-2 regression weights, all three seeds including the collapsed K562 9C seed.
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!='4.44.2':
 assert 'transformers' not in sys.modules,'Restart the Kaggle session: Transformers was imported before pinning.'
 subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2'],check=True,timeout=900)
from transformers import AutoTokenizer,AutoModel
import transformers
assert transformers.__version__=='4.44.2'
def backbone():
 remote='zhihan1996/DNABERT-2-117M'
 tok=AutoTokenizer.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 try:enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 except (ValueError,TypeError):
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=REVISION)
  enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True,config=cfg)
 import importlib
 mod=importlib.import_module(type(enc).__module__)
 assert hasattr(mod,'BertUnpadSelfAttention') and hasattr(mod,'flash_attn_qkvpacked_func')
 mod.flash_attn_qkvpacked_func=None
 return tok,enc

class Regressor(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc
  self.dropout=nn.Dropout(.1);self.head=nn.Linear(int(enc.config.hidden_size),1)
 def forward(self,b):
  output=self.encoder(**b);h=getattr(output,'last_hidden_state',None)
  if h is None:h=output[0]
  mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
  pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
  return self.head(self.dropout(pooled)).squeeze(-1)

def tokenize(tok,seqs):
 b=tok(list(seqs),padding=True,add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['dnabert2_max_tokens'],b['input_ids'].shape
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}

def bert_predict(net,tok,seqs,mean,scale):
 net.eval();out=[]
 with torch.inference_mode():
  for i in range(0,len(seqs),24):
   batch=seqs[i:i+24]
   x=tokenize(tok,batch);xr=tokenize(tok,[rc(s) for s in batch])
   with torch.autocast('cuda',dtype=torch.float16):
    p=net(x).float();q=net(xr).float()
   out.extend(((((p+q)/2)*scale)+mean).cpu().numpy())
 return np.asarray(out,dtype=float)

for tag in PROTOCOLS:
 for cell in CELLS:
  for seed in SEEDS:
   arm=f'dnabert2_seed_{seed}'
   if done(tag,cell,arm) is not None:
    print('REUSED DNABERT-2',tag,cell,seed,flush=True);continue
   path,temporary,digest=checkpoint_file(tag,cell,arm)
   ckpt=torch.load(path,map_location='cpu',weights_only=True)
   assert ckpt['lock']==LOCKS[tag] and ckpt['seed']==seed and ckpt['cell_line']==cell
   assert ckpt['revision']==REVISION
   if temporary:path.unlink()
   tok,enc=backbone();net=Regressor(enc).to(DEVICE)
   net.load_state_dict(ckpt['state_dict'],strict=True)
   values=bert_predict(net,tok,external.sequence.tolist(),ckpt['target_mean'],ckpt['target_scale'])
   save_score(tag,cell,arm,values,digest)
   print('SAVED DNABERT-2',tag,cell,seed,flush=True)
   del values,ckpt,tok,enc,net;gc.collect();torch.cuda.empty_cache()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 75.2 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

...

SAVED DNABERT-2 9c K562 20260930


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SAVED DNABERT-2 9c K562 20260931


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SAVED DNABERT-2 9c K562 20260932


In [ ]:
# 6 — Only after all frozen model scores exist: external descriptive metrics and paired bootstrap.
from scipy.stats import pearsonr
all_scores={}
for tag in PROTOCOLS:
 for cell in CELLS:
  for arm in ARMS:
   s=done(tag,cell,arm)
   assert s is not None,'Missing saved predictions: '+str((tag,cell,arm))
   all_scores[tag,cell,arm]=s
  for family in ('cnn','dnabert2'):
   all_scores[tag,cell,f'{family}_three_seed_mean']=np.stack(
    [all_scores[tag,cell,f'{family}_seed_{seed}'] for seed in SEEDS]).mean(axis=0)
print('All 32 individual model scores authenticated; 8 ensemble vectors calculated.')

strata=['DHS_natural','Malinois_natural','AdaLead','FastSeqProp','Simulated_Annealing']
metrics=[];rows=[]
for tag in PROTOCOLS:
 for cell in CELLS:
  truth=external[f'{cell}_l2fc'].to_numpy(dtype=float)
  for arm in ARMS+('cnn_three_seed_mean','dnabert2_three_seed_mean'):
   scores=all_scores[tag,cell,arm]
   for origin in strata:
    ix=np.flatnonzero(external.origin.eq(origin).to_numpy())
    y=truth[ix];pred=scores[ix]
    metrics.append({'protocol':tag,'cell_line':cell,'origin':origin,'primary':origin==CONFIG['primary_origin'],
     'model':arm,'n':len(ix),'spearman':float(spearmanr(y,pred).statistic),
     'pearson':float(pearsonr(y,pred).statistic),'prediction_std':float(np.std(pred))})
   rows.append(pd.DataFrame({'record_id':external.record_id,'protocol':tag,'cell_line':cell,
    'model':arm,'origin':external.origin,'activity':truth,'prediction':scores}))
predictions=pd.concat(rows,ignore_index=True)
metric=pd.DataFrame(metrics)
assert np.isfinite(metric[['spearman','pearson','prediction_std']]).all().all()
print('PRIMARY: independent DHS-natural sequences, no shared exact 31-mer with Phase 9')
print(metric[(metric.primary)&(metric.model.isin(['gc_cpg_ridge','spaced6_4_ridge','cnn_three_seed_mean','dnabert2_three_seed_mean']))].sort_values(
 ['cell_line','protocol','spearman'],ascending=[True,True,False]).to_string(index=False))

rng=np.random.default_rng(CONFIG['bootstrap_seed'])
primary_ix=np.flatnonzero(external.origin.eq(CONFIG['primary_origin']).to_numpy())
boot=[]
def paired_ci(y,a,b):
 diffs=[];n=len(y)
 for rep in range(CONFIG['bootstrap_repeats']):
  ix=rng.integers(0,n,n)
  diffs.append(float(spearmanr(y[ix],a[ix]).statistic-spearmanr(y[ix],b[ix]).statistic))
 return float(np.quantile(diffs,.025)),float(np.quantile(diffs,.975))
for cell in CELLS:
 y=external[f'{cell}_l2fc'].to_numpy(dtype=float)[primary_ix]
 for tag in PROTOCOLS:
  a=all_scores[tag,cell,'dnabert2_three_seed_mean'][primary_ix]
  b=all_scores[tag,cell,'spaced6_4_ridge'][primary_ix]
  lo,hi=paired_ci(y,a,b)
  boot.append({'cell_line':cell,'contrast':f'{tag} DNABERT-2 mean minus {tag} spaced ridge',
   'n':len(y),'spearman_difference':float(spearmanr(y,a).statistic-spearmanr(y,b).statistic),
   'lower_95':lo,'upper_95':hi,'repeats':CONFIG['bootstrap_repeats'],'resampling_unit':'unique external DNA sequence'})
 a=all_scores['9c',cell,'dnabert2_three_seed_mean'][primary_ix]
 b=all_scores['9b',cell,'dnabert2_three_seed_mean'][primary_ix]
 lo,hi=paired_ci(y,a,b)
 boot.append({'cell_line':cell,'contrast':'9C DNABERT-2 mean minus 9B DNABERT-2 mean',
  'n':len(y),'spearman_difference':float(spearmanr(y,a).statistic-spearmanr(y,b).statistic),
  'lower_95':lo,'upper_95':hi,'repeats':CONFIG['bootstrap_repeats'],'resampling_unit':'unique external DNA sequence'})
bootstrap=pd.DataFrame(boot)
print(bootstrap.to_string(index=False))


All 32 individual model scores authenticated; 8 ensemble vectors calculated.
PRIMARY: independent DHS-natural sequences, no shared exact 31-mer with Phase 9
protocol cell_line      origin  primary                    model    n  spearman  pearson  prediction_std
      9b     HepG2 DHS_natural     True dnabert2_three_seed_mean 8399  0.457040 0.480335        0.332673
      9b     HepG2 DHS_natural     True          spaced6_4_ridge 8399  0.417550 0.425110        0.346178
      9b     HepG2 DHS_natural     True      cnn_three_seed_mean 8399  0.321506 0.348865        0.205265
      9b     HepG2 DHS_natural     True             gc_cpg_ridge 8399  0.101162 0.101483        0.110194
      9c     HepG2 DHS_natural     True dnabert2_three_seed_mean 8399  0.466125 0.488104        0.328010
      9c     HepG2 DHS_natural     True          spaced6_4_ridge 8399  0.420205 0.426938        0.345117
      9c     HepG2 DHS_natural     True      cnn_three_seed_mean 8399  0.351837 0.379921        0.225422
   

In [ ]:
# 7 — Package the frozen external source audit, complete predictions and provenance.
ART=WORK/'results';ART.mkdir(exist_ok=True)
files={}
def write_csv(name,frame):
 p=ART/name;frame.to_csv(p,index=False,float_format='%.12g');files[name]=p
write_csv('phase10_external_predictions.csv',predictions)
write_csv('phase10_external_metrics.csv',metric)
write_csv('phase10_external_bootstrap.csv',bootstrap)
write_csv('phase10_external_frozen_set.csv',external[cols])
inc=ART/'phase10_inclusion_audit.json';inc.write_text(json.dumps(AUDIT,indent=2,sort_keys=True)+'\n');files[inc.name]=inc
report={'protocol_sha256':LOCK,'config':CONFIG,'source_url':SOURCE_URLS,
 'source_sha256':EXTERNAL_HASH,'frozen_external_sha256':FROZEN_SHA,
 'phase9_result_sha256':RESULT_SHA,'phase9_checkpoint_sha256':{
  tag:REPORT[tag]['checkpoints'] for tag in PROTOCOLS},
 'phase9a_manifest_sha256':INPUT_HASH,'result_sha256':{name:fsha(p) for name,p in files.items()},
 'source_publication':'Gosai et al. Nature 2024 doi:10.1038/s41586-024-08070-z Supplementary Table 12',
 'limitations':['Episomal CODA assay and ENCODE lentiMPRA have incomparable raw scales.',
  'DHS-natural sequences come from a selected experimental library, not genome-wide sampling.',
  'Natural hg38 intervals were screened against Phase 9A, but synthetic records have no genomic coordinates.',
  'An exact 31-mer screen is conservative but cannot exclude all homology.',
  'Synthetic designs may be related and were selected with a different sequence model.',
  'K562 Phase 9C seed 20260930 collapsed on its internal validation; included in fixed mean.']}
p=ART/'phase10_provenance.json';p.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n');files[p.name]=p
OUT=Path('/kaggle/working/enhancer_phase10_external_transfer_results.zip')
with zipfile.ZipFile(OUT,'w',allowZip64=True) as z:
 for name,p in files.items():z.write(p,name,compress_type=zipfile.ZIP_DEFLATED)
with zipfile.ZipFile(OUT) as z:
 assert z.testzip() is None
 for name,digest in report['result_sha256'].items():assert sha(z.read(name))==digest
print('READY:',OUT,'SHA256',fsha(OUT),'bytes',OUT.stat().st_size)
from IPython.display import display,FileLink
os.chdir('/kaggle/working');display(FileLink(OUT.name))


READY: /kaggle/working/enhancer_phase10_external_transfer_results.zip SHA256 60a4612601c4f79fac1ae55ba53712f58ff4207644149de20432d7a0b66da81a bytes 38686027


/kaggle/working/enhancer_phase10_external_transfer_results.zip